# core

> Fill in a module description here

In [ ]:
#| default_exp core

In [ ]:
#| hide
#| eval: false

import plotly.io as pio
import numpy as np

In [ ]:
#| hide
#| eval: false
pio.renderers.default = "png"

In [ ]:
#| hide

from fhemb import setup_logging
from fhemb.piece import Piece
from fhemb.utils.factories import wfactory, wfactories, pca_factory

DEBUG:fhemb.config.settings:Loading environment from /Users/radned/.config/fhemb/.env.paths
DEBUG:fhemb.config.settings:Loading environment from /Users/radned/.config/fhemb/.env.db


In [ ]:
#| hide
#| eval: false

setup_logging(
    force=True,
    module_levels={
        "mixins": "WARNING",
        "piece": "INFO",
        "dbms": "INFO",
        "embedding": "INFO",
        "tutils": "INFO",
        "cutils": "INFO",
        "wdecomposition": "INFO",
        "wrepresentation": "INFO",
    },
)

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
def foo(): pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

Specify a segment of interest

> Instantiate the `Piece` metadata class, which stores all relevant segment information, including:

- the opera title,
- the time interval (as a frame range),
- subject intervals, where each subject is associated with a tracked heatmap time series.

In [ ]:
#| eval: true

dg0 = Piece(              
    title='Don Giovanni',
    time_interval=(14200,15200), 
    subjs=[(40, 42), 50, (65,68)]
) 

Request data stored in a server-side database.

> Available data fields include:

- region of interest: `roi`, `roi_t`
- face bounding box: `facebbox`, `facebbox_t`
- face position: `faceposition_t`

`roi` vs `roi_t` and `facebbox` vs `facebbox_t` differ slightly because they come from different tracking methods.   

In [ ]:
#| eval: false

dg0.facebbox_t

 [INFO] piece._load_from_db: Loading facebbox_t from the database
 [INFO] piece._fetch_from_DB: Fetching data by query SELECT subj40, subj41, subj42, subj50, subj65, subj66, subj67, subj68 FROM dg.faceposition_t WHERE frame_number >= 14200 AND frame_number < 15200 ORDER BY frame_number
/Users/radned/.pyenv/versions/p311.fhemb/lib/python3.11/site-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
/Users/radned/.pyenv/versions/p311.fhemb/lib/python3.11/site-packages/paramiko/transport.py:253: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,
 [INFO] dbms.data_query: Fetching time series of [

<fhemb.dbms.db.Datasource>

## `dg0.facebbox_t` API

`dg0.facebbox_t` is a `Datasource` object (`fhemb.dbms.db.Datasource`) used to access tracked face-bounding-box time-series data and derived embeddings.

### Main methods

- `project(subjs='all', features=None, heatmap=None, fvector=None, fbands=None, pcfactory=None, wdfactory=None) -> Embedding`  
  Project selected data into an `Embedding` (optionally with PCA and/or wavelet decomposition).

- `create_pca_embedding(factory)`  
  Build a PCA-based embedding using a `ConcretePcaFactory`.

- `create_concat_embedding(embeddings, pca_key=None, wd_key=None, wr_key=None) -> Embedding`  
  Concatenate multiple embeddings into one combined embedding.

### Lag/correlation analysis

- `lagged_correlations_df(subjs, features, fbands, twin, max_lag, step, wdfactory=None, pcfactory=None, bbootstrap=False, n_jobs=4)`  
  Compute lagged correlations and return result dataframes.

- `plot_lagged_correlations(subjs, features, fbands=[], twin=50, max_lag=200, step=25, wdfactory=None, pcfactory=None, bbootstrap=False, n_jobs=4, **kwargs)`  
  Visualize rolling time-lagged cross-correlations.

- `lagged_correlation_graph(subjs=[], features=[], lag=0, fbands=[], factory=None, twin=50, step=25, max_lag=200, bbootstrap=False, n_jobs=4)`  
  Plot lagged-correlation graphs per subject/feature selection.

### Distribution/summary

- `plot_pdf(subj, secs, bw_method=1)`  
  Plot a PDF estimate at selected time points.

### Subject validation helper

- `get_subjects(subjs, sort=True) -> list[str]`  
  Normalize and validate subject selection (`'all'` or list of indices).

### Useful properties

- `facesizes_t` *(readonly)*  
- `rsfacebbox_t` *(readonly)*  
- `features_ts`, `percentiles_ts`, `binheights_ts`, `binedges` *(inherited readonly)*  
- `sr = 25` *(sampling rate, class attribute)*

### Class method

- `load_attribute(name=None)`  
  Load an attribute from the DB or derive it from existing attributes.

In [ ]:
#| eval: false

dg0.facebbox_t.rsfacebbox_t.features_tint['subj50'][10].shape


 [INFO] piece._derive_features: Deriving binheights_ts, binedges, percentiles_ts, and features_ts


 [INFO] piece._derive_tracked: Deriving rsfacebbox_t and facesizes_t


(60, 40)

In [ ]:
#eval: false

dg0.face_t.facesizes_t.features_tint['subj50'][10]


476

In [ ]:
#.project(features=['skew'], subjs=[65], fbands=[0,1,2], wdfactory=wfactory('db4',8)).plot_signal(subjs=[65])